# 04 — Two-Stage Evaluation Pipeline (Colab -> Local)

This notebook is intentionally split into two environment-independent sections:

1. **Section 1 (Colab, GPU models):** Prepare data and run GPU-bound local models (Qwen base, QLoRA, XLM-R).
2. **Manual sync step:** Copy the updated `results/` directory from Drive to your local machine.
3. **Section 2 (Local, CPU/API):** Run baseline + API evaluations and regenerate final aggregates/figures.

Use each section independently with its own setup cell.

In [ ]:
# Section 1 (Colab) - Setup
from google.colab import drive
drive.mount('/content/drive')

import os
PROJECT_ROOT = '/content/drive/MyDrive/qwen-sentiment'
assert os.path.isdir(PROJECT_ROOT), f'Project not found at {PROJECT_ROOT}'
os.chdir(PROJECT_ROOT)
print(f'✅ Working directory: {os.getcwd()}')

# Colab dependencies for GPU/local-model evaluation
%pip install -q unsloth transformers datasets accelerate bitsandbytes peft trl
%pip install -q groq google-genai python-dotenv
%pip install -q -r requirements-colab.txt

In [ ]:
# Section 1 (Colab) - Imports and variables (GPU/local models only)
import json
import os
from pathlib import Path

import torch
assert torch.cuda.is_available(), 'No GPU detected. Use T4 runtime.'
print(f'✅ GPU: {torch.cuda.get_device_name(0)}')

from src.dataset import load_subsample_indices
from src.infer_local import load_base_model, load_finetuned_model, run_local_evaluation
from src.infer_xlmr import load_xlmr_model, run_xlmr_evaluation

SEEDS = [42, 123, 456]
LANGUAGES = ['english', 'hindi', 'spanish', 'arabic']
EVAL_ROOT = Path('results/eval_scores')
CACHE_ROOT = Path('results/api_cache')
SUBSAMPLE_PATH = EVAL_ROOT / 'subsample_indices.json'

def atomic_copy_json(src: Path, dst: Path):
    dst.parent.mkdir(parents=True, exist_ok=True)
    tmp = dst.with_suffix(dst.suffix + '.tmp')
    with open(src, 'r', encoding='utf-8') as fsrc:
        data = json.load(fsrc)
    with open(tmp, 'w', encoding='utf-8') as fdst:
        json.dump(data, fdst, indent=2, ensure_ascii=False)
    os.replace(str(tmp), str(dst))

print('✅ Section 1 imports ready')

In [ ]:
# Section 1 - Phase 1: Subsample backup/delete/regenerate (n=870, seed=42)
from src.dataset import load_all_splits, save_subsample_indices

backup_path = EVAL_ROOT / 'subsample_indices_n200_backup.json'
if SUBSAMPLE_PATH.exists() and not backup_path.exists():
    atomic_copy_json(SUBSAMPLE_PATH, backup_path)
    print(f'✅ Backup created: {backup_path}')

if SUBSAMPLE_PATH.exists():
    SUBSAMPLE_PATH.unlink()
    print(f'🗑️ Deleted: {SUBSAMPLE_PATH}')

all_data = load_all_splits(LANGUAGES)
test_datasets = {lang: all_data[lang]['test'] for lang in LANGUAGES}
_ = save_subsample_indices(
    test_datasets,
    n_samples=870,
    seed=42,
    save_path=str(SUBSAMPLE_PATH),
)

indices_data = load_subsample_indices(str(SUBSAMPLE_PATH))
print('✅ Regenerated subsample indices at n=870')

In [ ]:
# Section 1 - Phase 2: Migrate existing seed-42 metrics and QLoRA cache
import shutil

seed42_eval_dir = EVAL_ROOT / 'seed_42'
seed42_eval_dir.mkdir(parents=True, exist_ok=True)

for p in EVAL_ROOT.glob('*.json'):
    if (p.name == 'subsample_indices.json'
            or p.name.endswith('_parse_failures.json')
            or 'backup' in p.name
            or p.name == 'hindi_script_composition.json'):
        continue
    target = seed42_eval_dir / p.name
    if not target.exists():
        shutil.move(str(p), str(target))
        print(f'Moved metric: {p} -> {target}')

for lang in LANGUAGES:
    src = CACHE_ROOT / 'qwen25_finetuned' / lang / 'predictions.json'
    dst = CACHE_ROOT / 'qwen25_finetuned' / 'seed_42' / lang / 'predictions.json'
    if src.exists() and not dst.exists():
        dst.parent.mkdir(parents=True, exist_ok=True)
        shutil.move(str(src), str(dst))
        print(f'Moved cache: {src} -> {dst}')

print('✅ Seed-42 migration complete')

In [ ]:
# Section 1 - Build test subsets from canonical regenerated indices
from src.dataset import get_stratified_subsample, load_all_splits

data = load_all_splits(LANGUAGES)
indices_data = load_subsample_indices(str(SUBSAMPLE_PATH))

test_subsets = {}
for lang in LANGUAGES:
    sub_indices = indices_data['indices'][lang]
    test_subsets[lang] = get_stratified_subsample(
        data[lang]['test'],
        n_samples=870,
        indices_override=sub_indices,
    )
    print(f'{lang}: {len(test_subsets[lang])} samples')

print(f'\nTotal evaluation samples: {sum(len(v) for v in test_subsets.values())}')

In [ ]:
# Section 1 - Phase 3 (partial): Qwen2.5 base only
print('Phase 3 (Colab partial): qwen25_base')

from src.evaluate import save_results
from src.infer_local import load_base_model, run_local_evaluation

model_base, tok_base = load_base_model('unsloth/Qwen2.5-3B-Instruct')
for lang in LANGUAGES:
    original_indices = indices_data['indices'][lang]
    metrics = run_local_evaluation(
        model_base, tok_base, test_subsets[lang], original_indices,
        model_name='qwen25_base', language=lang, seed=42,
    )
    for s in [123, 456]:
        save_results(metrics, 'qwen25_base', lang, seed=s)
del model_base, tok_base

print('✅ Section 1 Phase 3 partial complete (Qwen base only)')

## Section 1 (Colab) - Phases 4 and 5

Phase 4 evaluates QLoRA checkpoints per seed.
Phase 5 evaluates XLM-R checkpoints per seed.

In [ ]:
# Section 1 - Phase 4: QLoRA evaluation across seeds
for seed in SEEDS:
    print('\n' + '=' * 70)
    print(f'QLoRA evaluation — seed {seed}')
    print('=' * 70)

    repo_id = f'Mrigank005/SLM-sentiment-crosslingual-seed-{seed}'
    model_ft, tok_ft = load_finetuned_model(repo_id)

    for lang in LANGUAGES:
        original_indices = indices_data['indices'][lang]
        metrics = run_local_evaluation(
            model_ft, tok_ft, test_subsets[lang], original_indices,
            model_name='qwen25_finetuned', language=lang, seed=seed,
        )
        print(f"  {lang}: F1={metrics['f1_macro']:.4f}")

    del model_ft, tok_ft

print('✅ Phase 4 complete')

In [ ]:
# Section 1 - Phase 5: XLM-R evaluation across seeds
for seed in SEEDS:
    print('\n' + '=' * 70)
    print(f'XLM-R evaluation — seed {seed}')
    print('=' * 70)

    repo_id = f'Mrigank005/xlm-roberta-sentiment-crosslingual-seed-{seed}'
    model_xlmr, tok_xlmr = load_xlmr_model(repo_id)

    for lang in LANGUAGES:
        original_indices = indices_data['indices'][lang]
        metrics = run_xlmr_evaluation(
            model_xlmr, tok_xlmr, test_subsets[lang],
            original_indices=original_indices, language=lang, seed=seed,
        )
        print(f"  {lang}: F1={metrics['f1_macro']:.4f}")

    del model_xlmr, tok_xlmr

print('✅ Phase 5 complete')

## Section 2 (Local) - CPU/API Evaluations

Run this section only after syncing the updated `results/` directory from Colab to local.

In [ ]:
# Section 2 (Local) - Setup
import os
import sys
from pathlib import Path
from dotenv import load_dotenv

PROJECT_ROOT = Path(os.environ.get('PROJECT_ROOT', os.getcwd())).resolve()

# If launched from notebooks/ or another subfolder, walk upward to find src/.
if not (PROJECT_ROOT / 'src').is_dir():
    for candidate in [PROJECT_ROOT, *PROJECT_ROOT.parents]:
        if (candidate / 'src').is_dir():
            PROJECT_ROOT = candidate
            break

assert (PROJECT_ROOT / 'src').is_dir(), (
    f'Could not find project root with src/ from {os.getcwd()}. '
    'Set PROJECT_ROOT in .env if needed.'
)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

os.chdir(PROJECT_ROOT)
load_dotenv()

print(f'✅ Local working directory: {os.getcwd()}')
print(f'✅ src import path ready: {PROJECT_ROOT / "src"}')
print('✅ .env loaded (if present)')

In [ ]:
# Section 2 (Local) - Imports for API models, baselines, and aggregation
import json
import os
import sys
from pathlib import Path

# Make this cell resilient if run before setup cell.
_cwd = Path(os.getcwd()).resolve()
_root = Path(os.environ.get('PROJECT_ROOT', _cwd)).resolve()
if not (_root / 'src').is_dir():
    for candidate in [_cwd, *_cwd.parents]:
        if (candidate / 'src').is_dir():
            _root = candidate
            break
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

from src.dataset import load_all_splits, get_stratified_subsample, load_subsample_indices
from src.evaluate import majority_class_baseline, save_results, aggregate_seed_results
from src.infer_groq import KeyRotator, run_groq_evaluation
from src.infer_gemini import run_gemini_evaluation, RotatingGeminiClient

SEEDS = [42, 123, 456]
LANGUAGES = ['english', 'hindi', 'spanish', 'arabic']
EVAL_ROOT = Path('results/eval_scores')
CACHE_ROOT = Path('results/api_cache')
SUBSAMPLE_PATH = EVAL_ROOT / 'subsample_indices.json'

print('✅ Section 2 imports ready')

In [ ]:
# Section 2 (Local) - Rebuild test_subsets from synced subsample indices
assert SUBSAMPLE_PATH.exists(), f'Missing synced indices file: {SUBSAMPLE_PATH}'

data = load_all_splits(LANGUAGES)
indices_data = load_subsample_indices(str(SUBSAMPLE_PATH))

test_subsets = {}
for lang in LANGUAGES:
    sub_indices = indices_data['indices'][lang]
    test_subsets[lang] = get_stratified_subsample(
        data[lang]['test'],
        n_samples=870,
        indices_override=sub_indices,
    )
    print(f'{lang}: {len(test_subsets[lang])} samples')

print(f'\nTotal evaluation samples: {sum(len(v) for v in test_subsets.values())}')

In [ ]:
# Section 2 (Local) - Phase 3.1: Majority baseline
print('Phase 3.1 (Local): majority_baseline')

for lang in LANGUAGES:
    y_true = list(test_subsets[lang]['label'])
    _, metrics = majority_class_baseline(y_true)
    for seed in SEEDS:
        save_results(metrics, 'majority_baseline', lang, seed=seed)
    print(f"  {lang}: F1={metrics['f1_macro']:.4f}")

print('✅ Section 2 Phase 3.1 complete')

In [ ]:
# Section 2 (Local) - Phase 3.2: Llama70B (Groq)
print('Phase 3.2 (Local): llama70b')

groq_keys = [
    k for k in [
        os.environ.get('GROQ_API_KEY_1'),
        os.environ.get('GROQ_API_KEY_2'),
        os.environ.get('GROQ_API_KEY_3'),
        os.environ.get('GROQ_API_KEY_4'),
    ] if k
]

if groq_keys:
    rotator = KeyRotator(groq_keys)
    for lang in LANGUAGES:
        original_indices = indices_data['indices'][lang]
        metrics = run_groq_evaluation(rotator, test_subsets[lang], original_indices, lang, seed=42)
        for s in [123, 456]:
            save_results(metrics, 'llama70b', lang, seed=s)
        print(f"  {lang}: F1={metrics['f1_macro']:.4f}")
else:
    print('⚠️ No Groq keys found; skipping llama70b.')

print('✅ Section 2 Phase 3.2 complete')

In [ ]:
# Section 2 (Local) - Phase 3.3: Gemini 2.5 Pro
print('Phase 3.3 (Local): Gemini 2.5 Pro')

gemini_keys = [
    k for k in [
        os.environ.get('GOOGLE_API_KEY_1'),
        os.environ.get('GOOGLE_API_KEY_2'),
    ] if k
]

if gemini_keys:
    g_client = RotatingGeminiClient(gemini_keys)
    for lang in LANGUAGES:
        original_indices = indices_data['indices'][lang]
        metrics = run_gemini_evaluation(
            g_client,
            test_subsets[lang],
            original_indices,
            lang,
            seed=42,
        )
        for s in [123, 456]:
            save_results(metrics, 'gemini25pro', lang, seed=s)
        print(f"  {lang}: F1={metrics['f1_macro']:.4f}")
else:
    print('⚠️ No Gemini keys found; skipping Gemini 2.5 Pro.')

print('✅ Section 2 Phase 3.3 complete')

In [ ]:
# Section 2 (Local) - Phase 6: Aggregate metrics and regenerate figures/tables
from transformers import AutoTokenizer
from src.visualize import export_all_figures
from src.cost_analysis import compute_cost_table, plot_cost_vs_performance
from src.fertility import fertility_analysis, plot_fertility_vs_f1

MODELS = [
    'majority_baseline',
    'qwen25_base',
    'qwen25_finetuned',
    'llama70b',
    'gemini25pro',
    'xlm_roberta_base',
]

for model_name in MODELS:
    for lang in LANGUAGES:
        aggregate_seed_results(model_name, lang, seeds=SEEDS, results_dir='results/eval_scores/')

export_all_figures(results_dir='results', dpi=300, fmt='pdf')

# Explicitly regenerate cost_vs_f1.pdf
cost_df = compute_cost_table(
    results_dir='results/eval_scores/aggregated/',
    cache_dir='results/api_cache/',
    languages=LANGUAGES,
)
plot_cost_vs_performance(cost_df, save_path='results/figures/cost_vs_f1.pdf')

# Explicitly regenerate fertility_vs_f1.pdf
tokenizer = AutoTokenizer.from_pretrained('unsloth/Qwen2.5-3B-Instruct')
fertility_df = fertility_analysis(
    test_subsets,
    tokenizer,
    LANGUAGES,
    results_dir='results/eval_scores/aggregated/',
    model_name='qwen25_finetuned',
)
plot_fertility_vs_f1(fertility_df, save_path='results/figures/fertility_vs_f1.pdf')

print('✅ Section 2 Phase 6 complete: aggregation + figures regenerated')